# Behavior Diagnostics Next Steps

Next-step diagnostics for the gridworld action-commitment behavior manifold experiment.

This notebook is offline by default. It does not call Together AI or any other remote scoring API. It may prepare prompt-only or mismatched prompt/CoT parquet inputs and print the exact commands needed to score them as an explicit opt-in step.

## 0. Setup and Load

In [ ]:

from pathlib import Path
import json
import math
import os
import re
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scipy  # noqa: F401
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from sklearn.metrics import pairwise_distances
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({
    "figure.figsize": (7.5, 5.0),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

START_CWD = Path.cwd()
workspace_fallback = Path("/Users/wws/Desktop/SPAR_code")
candidate_roots = [Path.cwd(), *Path.cwd().parents, workspace_fallback]
ROOT = None
for candidate in candidate_roots:
    if (candidate / "outputs").exists() and (candidate / "scripts").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    ROOT = Path.cwd().resolve()
    print(f"WARNING: could not locate repo root from {START_CWD}; using {ROOT}")
elif ROOT != Path.cwd().resolve():
    os.chdir(ROOT)
    print(f"Changed working directory from {START_CWD} to {ROOT}")
FIG_DIR = Path("figures/diagnostics")
OUT_DIR = Path("outputs/diagnostics")
Q_EXAMPLE_DIR = FIG_DIR / "q_curves_examples"
for path in [FIG_DIR, OUT_DIR, Q_EXAMPLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PREFIX_METRICS_PATH = Path("outputs/prefix_commitment_metrics.parquet")
STEP_SUMMARY_PATH = Path("outputs/step_commitment_summary.parquet")
MANIFOLD_PATH = Path("outputs/behavior_manifold.parquet")
PREFIXES_PATH = Path("outputs/prefixes.parquet")
CANONICAL_PATH = Path("outputs/canonical_steps.parquet")

ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]
Q_COLS = [f"q_{a}" for a in ACTIONS]
WARNINGS = []


def warn(msg):
    WARNINGS.append(str(msg))
    print(f"WARNING: {msg}")


def load_required(path):
    if not path.exists():
        raise FileNotFoundError(f"Required file is missing: {path}")
    print(f"Loading {path}")
    return pd.read_parquet(path)


def load_optional(path):
    if not path.exists():
        warn(f"Optional file missing, skipping dependent diagnostics: {path}")
        return None
    print(f"Loading optional {path}")
    return pd.read_parquet(path)


def ensure_decision_id(df):
    if df is None:
        return None
    if "decision_id" not in df.columns:
        if {"trajectory_id", "step_id"}.issubset(df.columns):
            df = df.copy()
            df["decision_id"] = df["trajectory_id"].astype(str) + "::" + df["step_id"].astype(str)
        else:
            warn("Cannot create decision_id because trajectory_id/step_id are missing")
    return df


def q_for_action(df, action_col="agent_action", prefix="q_"):
    values = []
    for _, row in df.iterrows():
        action = row.get(action_col)
        col = f"{prefix}{action}"
        values.append(row[col] if col in row.index and pd.notna(row[col]) else np.nan)
    return pd.Series(values, index=df.index, dtype="float64")


def compute_entropy_commitment(df):
    q = df[Q_COLS].astype(float).clip(lower=0).to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        entropy = -(np.where(q > 0, q * np.log(q), 0)).sum(axis=1)
    return 1 - entropy / np.log(4)


def safe_name(value, max_len=120):
    text = str(value)
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")
    return text[:max_len] or "item"


def save_fig(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path}")
    return path


def save_q_fig(fig, filename):
    path = Q_EXAMPLE_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path}")
    return path


def standard_error(x):
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return np.nan
    return float(x.std(ddof=1) / np.sqrt(len(x)))


def final_prefix_rows(df):
    if df.empty:
        return df.copy()
    return df.sort_values(["decision_id", "prefix_token_index"]).groupby("decision_id", as_index=False).tail(1)


def first_per_decision(df):
    if df.empty:
        return df.copy()
    return df.sort_values(["decision_id", "prefix_token_index"]).groupby("decision_id", as_index=False).head(1)


def binned_per_step_curve(data, value_col, progress_col="progress", n_bins=20):
    required = ["decision_id", progress_col, value_col]
    missing = [c for c in required if c not in data.columns]
    if missing:
        warn(f"Cannot bin {value_col}; missing columns: {missing}")
        return pd.DataFrame(), pd.DataFrame()
    work = data[required].dropna().copy()
    if work.empty:
        return pd.DataFrame(), pd.DataFrame()
    work[progress_col] = work[progress_col].astype(float).clip(0, 1)
    edges = np.linspace(0, 1, n_bins + 1)
    work["progress_bin"] = pd.cut(work[progress_col], bins=edges, labels=False, include_lowest=True)
    work["progress_bin"] = work["progress_bin"].astype("Int64")

    per_decision = (
        work.groupby(["decision_id", "progress_bin"], observed=False)[value_col]
        .mean()
        .reset_index()
    )
    summary = (
        per_decision.groupby("progress_bin", observed=False)[value_col]
        .agg(mean="mean", std="std", n_steps="count")
        .reset_index()
    )
    summary["sem"] = summary["std"] / np.sqrt(summary["n_steps"].replace(0, np.nan))

    counts = (
        work.groupby("progress_bin", observed=False)
        .agg(prefix_points=(value_col, "size"), decision_steps=("decision_id", "nunique"))
        .reset_index()
    )

    all_bins = pd.DataFrame({"progress_bin": np.arange(n_bins)})
    for frame in [summary, counts]:
        if not frame.empty:
            frame["progress_bin"] = frame["progress_bin"].astype(int)
    summary = all_bins.merge(summary, on="progress_bin", how="left")
    counts = all_bins.merge(counts, on="progress_bin", how="left").fillna({"prefix_points": 0, "decision_steps": 0})
    for frame in [summary, counts]:
        frame["bin_left"] = edges[frame["progress_bin"].to_numpy()]
        frame["bin_right"] = edges[frame["progress_bin"].to_numpy() + 1]
        frame["progress_mid"] = (frame["bin_left"] + frame["bin_right"]) / 2
    return summary, counts


def plot_curve(summary, value_name, ylabel, title, filename, ylim=None):
    if summary.empty or summary["mean"].dropna().empty:
        warn(f"Skipping {filename}; no data")
        return None
    fig, ax = plt.subplots()
    x = summary["progress_mid"].to_numpy(dtype=float)
    y = summary["mean"].to_numpy(dtype=float)
    sem = summary["sem"].fillna(0).to_numpy(dtype=float)
    ax.plot(x, y, marker="o", linewidth=2)
    ax.fill_between(x, y - sem, y + sem, alpha=0.2)
    ax.set_xlabel("progress")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_xlim(0, 1)
    return save_fig(fig, filename)


def plot_counts(counts, title, filename):
    if counts.empty:
        warn(f"Skipping {filename}; no count data")
        return None
    fig, ax = plt.subplots()
    width = 0.04 if len(counts) > 15 else 0.08
    ax.bar(counts["progress_mid"], counts["prefix_points"], width=width, alpha=0.75, label="prefix points")
    ax.set_xlabel("progress")
    ax.set_ylabel("prefix points")
    ax2 = ax.twinx()
    ax2.plot(counts["progress_mid"], counts["decision_steps"], color="black", marker="o", linewidth=1.5, label="decision steps")
    ax2.set_ylabel("unique decision steps")
    ax.set_title(title)
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc="best")
    ax.set_xlim(0, 1)
    return save_fig(fig, filename)


def categorical_scatter(df, x, y, color_col, title, filename):
    if df.empty or x not in df.columns or y not in df.columns or color_col not in df.columns:
        warn(f"Skipping {filename}; missing PCA/categorical data")
        return None
    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    for label, sub in df.groupby(color_col, dropna=False):
        ax.scatter(sub[x], sub[y], s=10, alpha=0.65, label=str(label))
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend(markerscale=2, fontsize=8, frameon=True)
    return save_fig(fig, filename)


def continuous_scatter(df, x, y, color_col, title, filename, cmap="viridis"):
    if df.empty or x not in df.columns or y not in df.columns or color_col not in df.columns:
        warn(f"Skipping {filename}; missing PCA/continuous data")
        return None
    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    sc = ax.scatter(df[x], df[y], c=df[color_col], s=10, alpha=0.7, cmap=cmap)
    fig.colorbar(sc, ax=ax, label=color_col)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    return save_fig(fig, filename)


def action_distribution(series):
    return series.value_counts(dropna=False).reindex(ACTIONS).fillna(0).astype(int)

print(f"scipy available: {SCIPY_AVAILABLE}")
print(f"sklearn available: {SKLEARN_AVAILABLE}")


In [ ]:

prefix_df = load_required(PREFIX_METRICS_PATH)
step_df = load_required(STEP_SUMMARY_PATH)
manifold_df = load_required(MANIFOLD_PATH)
prefixes_optional = load_optional(PREFIXES_PATH)
canonical_optional = load_optional(CANONICAL_PATH)

prefix_df = ensure_decision_id(prefix_df)
step_df = ensure_decision_id(step_df)
manifold_df = ensure_decision_id(manifold_df)
prefixes_optional = ensure_decision_id(prefixes_optional)
canonical_optional = ensure_decision_id(canonical_optional)

# Enrich scored prefixes with metadata from the full prefix table when present.
# The scoring subset may not carry first_action_mention_token_index or analysis_prefix_text.
if prefixes_optional is not None:
    enrich_cols = [
        "prefix_id", "decision_id", "trajectory_id", "source_file", "step_id", "prefix_token_index",
        "prompt_text", "analysis_prefix_text", "analysis_prefix_token_ids_json",
        "first_action_mention_token_index", "first_final_action_mention_token_index",
        "num_analysis_tokens", "progress", "agent_action",
    ]
    enrich_cols = [c for c in enrich_cols if c in prefixes_optional.columns]
    enrich = prefixes_optional[enrich_cols].drop_duplicates(subset=["prefix_id"] if "prefix_id" in enrich_cols else ["decision_id", "prefix_token_index"])
    merge_key = "prefix_id" if "prefix_id" in prefix_df.columns and "prefix_id" in enrich.columns else None
    if merge_key:
        add_cols = [c for c in enrich.columns if c != merge_key and c not in prefix_df.columns]
        if add_cols:
            prefix_df = prefix_df.merge(enrich[[merge_key] + add_cols], on=merge_key, how="left")
            print(f"Enriched prefix metrics with columns from prefixes.parquet: {add_cols}")
    else:
        key_cols = ["decision_id", "prefix_token_index"]
        if set(key_cols).issubset(prefix_df.columns) and set(key_cols).issubset(enrich.columns):
            add_cols = [c for c in enrich.columns if c not in key_cols and c not in prefix_df.columns]
            if add_cols:
                prefix_df = prefix_df.merge(enrich[key_cols + add_cols], on=key_cols, how="left")
                print(f"Enriched prefix metrics with columns from prefixes.parquet: {add_cols}")

# Recompute core metrics defensively if any are absent.
if "top_action" not in prefix_df.columns and set(Q_COLS).issubset(prefix_df.columns):
    prefix_df["top_action"] = prefix_df[Q_COLS].astype(float).idxmax(axis=1).str.replace("q_", "", regex=False)
if "commitment_to_final" not in prefix_df.columns and set(Q_COLS + ["agent_action"]).issubset(prefix_df.columns):
    prefix_df["commitment_to_final"] = q_for_action(prefix_df)
if "entropy_commitment" not in prefix_df.columns and set(Q_COLS).issubset(prefix_df.columns):
    prefix_df["entropy_commitment"] = compute_entropy_commitment(prefix_df)
if "margin_commitment" not in prefix_df.columns and set(Q_COLS).issubset(prefix_df.columns):
    sorted_q = np.sort(prefix_df[Q_COLS].astype(float).to_numpy(), axis=1)
    prefix_df["margin_commitment"] = sorted_q[:, -1] - sorted_q[:, -2]

required_prefix_cols = [
    "prefix_id", "trajectory_id", "step_id", "prefix_token_index", "progress", "agent_action",
    "q_UP", "q_DOWN", "q_LEFT", "q_RIGHT", "top_action", "commitment_to_final",
    "entropy_commitment", "margin_commitment",
]
missing_required = [c for c in required_prefix_cols if c not in prefix_df.columns]
if missing_required:
    print("Missing required prefix columns:", missing_required)
else:
    print("All required prefix columns are present.")

for df_name, df in [("prefix_df", prefix_df), ("step_df", step_df), ("manifold_df", manifold_df)]:
    if "decision_id" not in df.columns:
        warn(f"{df_name} has no decision_id after setup")

prefix_counts = prefix_df.groupby("decision_id").size()
summary_rows = [
    {"metric": "prefix_points", "value": len(prefix_df)},
    {"metric": "decision_steps", "value": prefix_df["decision_id"].nunique()},
    {"metric": "trajectories", "value": prefix_df["trajectory_id"].nunique() if "trajectory_id" in prefix_df.columns else np.nan},
    {"metric": "prefixes_per_decision_min", "value": prefix_counts.min()},
    {"metric": "prefixes_per_decision_median", "value": prefix_counts.median()},
    {"metric": "prefixes_per_decision_mean", "value": prefix_counts.mean()},
    {"metric": "prefixes_per_decision_max", "value": prefix_counts.max()},
]
basic_summary = pd.DataFrame(summary_rows)
basic_summary.to_csv(OUT_DIR / "basic_dataset_summary.csv", index=False)

print("\nDataset summary")
display(basic_summary)
print("\nagent_action distribution")
display(action_distribution(prefix_df["agent_action"]).rename("prefix_points"))
print("\ntop_action distribution")
display(action_distribution(prefix_df["top_action"]).rename("prefix_points"))
print("\nPrefixes per decision")
display(prefix_counts.describe())


## 1. Reproduce Key Progress Curves With SEM and Counts

In [ ]:

N_BINS = 20
commit_summary, commit_counts = binned_per_step_curve(prefix_df, "commitment_to_final", n_bins=N_BINS)
entropy_summary, entropy_counts = binned_per_step_curve(prefix_df, "entropy_commitment", n_bins=N_BINS)

commit_summary = commit_summary.assign(metric="commitment_to_final")
entropy_summary = entropy_summary.assign(metric="entropy_commitment")
progress_curve_table = pd.concat([commit_summary, entropy_summary], ignore_index=True)
progress_curve_table.to_csv(OUT_DIR / "progress_curve_per_step_weighted.csv", index=False)

plot_curve(commit_summary, "commitment_to_final", "commitment to final action", "Commitment to final action over reasoning progress", "commitment_to_final_progress_sem_per_step.png", ylim=(0, 1))
plot_curve(entropy_summary, "entropy_commitment", "entropy commitment", "Entropy commitment over reasoning progress", "entropy_commitment_progress_sem_per_step.png", ylim=(0, 1))
plot_counts(commit_counts, "Progress-bin coverage: all prefixes", "progress_bin_counts_all_prefixes.png")

display(progress_curve_table.head())


## 2. Prompt-Only Baseline

In [ ]:

PROMPT_ONLY_PREFIX_PATH = OUT_DIR / "prompt_only_prefixes.parquet"
PROMPT_ONLY_SCORE_PATH = OUT_DIR / "prompt_only_action_scores.parquet"

prompt_only_scores = None
prompt_only_source = None
prompt_only_command = """python scripts/03_score_prefix_actions_together.py \\
  --input outputs/diagnostics/prompt_only_prefixes.parquet \\
  --output outputs/diagnostics/prompt_only_action_scores.parquet \\
  --model_name openai/gpt-oss-20b \\
  --api_key_env TOGETHER_API_KEY \\
  --scoring_method echo_teacher_forced \\
  --candidate_mode bare_action \\
  --temperature 0.0 \\
  --top_p 1.0 \\
  --logprobs 20 \\
  --max_tokens 1 \\
  --save_every 50 \\
  --resume"""

if "prefix_token_index" in prefix_df.columns and (prefix_df["prefix_token_index"] == 0).any():
    prompt_only_scores = first_per_decision(prefix_df[prefix_df["prefix_token_index"] == 0].copy())
    prompt_only_source = "existing prefix_token_index == 0 rows"
    print(f"Using {len(prompt_only_scores)} existing prefix_token_index == 0 rows as the prompt-only baseline.")
elif PROMPT_ONLY_SCORE_PATH.exists():
    prompt_only_scores = ensure_decision_id(pd.read_parquet(PROMPT_ONLY_SCORE_PATH))
    prompt_only_source = str(PROMPT_ONLY_SCORE_PATH)
    print(f"Loaded scored prompt-only baseline from {PROMPT_ONLY_SCORE_PATH}")
else:
    if prefixes_optional is not None and {"decision_id", "trajectory_id", "step_id", "prompt_text", "agent_action"}.issubset(prefixes_optional.columns):
        base = first_per_decision(prefixes_optional[prefixes_optional["decision_id"].isin(prefix_df["decision_id"].unique())].copy())
        prompt_only_prefixes = pd.DataFrame({
            "prefix_id": base["decision_id"].astype(str) + "::prompt_only",
            "trajectory_id": base["trajectory_id"],
            "source_file": base["source_file"] if "source_file" in base.columns else None,
            "step_id": base["step_id"],
            "decision_id": base["decision_id"],
            "prefix_token_index": 0,
            "num_analysis_tokens": base["num_analysis_tokens"] if "num_analysis_tokens" in base.columns else np.nan,
            "progress": 0.0,
            "prompt_text": base["prompt_text"],
            "analysis_prefix_text": "",
            "analysis_prefix_token_ids_json": "[]",
            "agent_action": base["agent_action"],
            "is_before_first_action_mention": True,
            "is_before_first_final_action_mention": True,
        })
        prompt_only_prefixes.to_parquet(PROMPT_ONLY_PREFIX_PATH, index=False)
        print(f"Created {PROMPT_ONLY_PREFIX_PATH} with {len(prompt_only_prefixes)} rows.")
    else:
        warn("Cannot create prompt-only prefixes because outputs/prefixes.parquet lacks required columns.")
    print("Score prompt-only prefixes explicitly with:")
    print(prompt_only_command)

prefix_with_prompt_baseline = None
prompt_only_summary = pd.DataFrame()
if prompt_only_scores is not None and set(Q_COLS + ["agent_action", "decision_id"]).issubset(prompt_only_scores.columns):
    prompt_only_scores = prompt_only_scores.copy()
    if "commitment_to_final" in prompt_only_scores.columns:
        prompt_only_scores["prompt_only_commitment_to_final"] = prompt_only_scores["commitment_to_final"]
    else:
        prompt_only_scores["prompt_only_commitment_to_final"] = q_for_action(prompt_only_scores)
    if "entropy_commitment" in prompt_only_scores.columns:
        prompt_only_scores["prompt_only_entropy_commitment"] = prompt_only_scores["entropy_commitment"]
    else:
        prompt_only_scores["prompt_only_entropy_commitment"] = compute_entropy_commitment(prompt_only_scores)

    po_cols = ["decision_id", "prompt_only_commitment_to_final", "prompt_only_entropy_commitment"]
    prefix_with_prompt_baseline = prefix_df.merge(prompt_only_scores[po_cols], on="decision_id", how="left")
    prefix_with_prompt_baseline["delta_commitment_to_final"] = prefix_with_prompt_baseline["commitment_to_final"] - prefix_with_prompt_baseline["prompt_only_commitment_to_final"]
    prefix_with_prompt_baseline["delta_entropy_commitment"] = prefix_with_prompt_baseline["entropy_commitment"] - prefix_with_prompt_baseline["prompt_only_entropy_commitment"]

    finals = final_prefix_rows(prefix_with_prompt_baseline)
    prompt_only_summary = pd.DataFrame([
        {"metric": "source", "value": prompt_only_source},
        {"metric": "decision_steps_with_prompt_only", "value": prompt_only_scores["decision_id"].nunique()},
        {"metric": "mean_prompt_only_commitment_to_final", "value": prompt_only_scores["prompt_only_commitment_to_final"].mean()},
        {"metric": "mean_final_prefix_commitment_to_final", "value": finals["commitment_to_final"].mean()},
        {"metric": "mean_final_minus_prompt_commitment", "value": (finals["commitment_to_final"] - finals["prompt_only_commitment_to_final"]).mean()},
        {"metric": "mean_prompt_only_entropy_commitment", "value": prompt_only_scores["prompt_only_entropy_commitment"].mean()},
    ])
    prompt_only_summary.to_csv(OUT_DIR / "prompt_only_baseline_summary.csv", index=False)

    fig, ax = plt.subplots()
    ax.hist(prompt_only_scores["prompt_only_commitment_to_final"].dropna(), bins=20, color="#4C78A8", alpha=0.85)
    ax.set_xlabel("prompt-only commitment to final action")
    ax.set_ylabel("decision steps")
    ax.set_title("Prompt-only commitment distribution")
    save_fig(fig, "prompt_only_commitment_histogram.png")

    base_summary, _ = binned_per_step_curve(prefix_with_prompt_baseline, "commitment_to_final", n_bins=N_BINS)
    fig, ax = plt.subplots()
    x = base_summary["progress_mid"].to_numpy(dtype=float)
    y = base_summary["mean"].to_numpy(dtype=float)
    sem = base_summary["sem"].fillna(0).to_numpy(dtype=float)
    ax.plot(x, y, marker="o", linewidth=2, label="prefix commitment")
    ax.fill_between(x, y - sem, y + sem, alpha=0.18)
    po_mean = prompt_only_scores["prompt_only_commitment_to_final"].mean()
    po_sem = standard_error(prompt_only_scores["prompt_only_commitment_to_final"])
    ax.errorbar([0], [po_mean], yerr=[0 if pd.isna(po_sem) else po_sem], marker="s", color="black", label="prompt-only")
    ax.set_xlabel("progress")
    ax.set_ylabel("commitment to final action")
    ax.set_ylim(0, 1)
    ax.set_title("Commitment over progress with prompt-only baseline")
    ax.legend()
    save_fig(fig, "commitment_progress_with_prompt_only.png")

    delta_summary, _ = binned_per_step_curve(prefix_with_prompt_baseline, "delta_commitment_to_final", n_bins=N_BINS)
    plot_curve(delta_summary, "delta_commitment_to_final", "delta commitment vs prompt-only", "Commitment gain over prompt-only baseline", "delta_commitment_over_progress.png")

    finals = final_prefix_rows(prefix_with_prompt_baseline)
    fig, ax = plt.subplots()
    ax.scatter(finals["prompt_only_commitment_to_final"], finals["commitment_to_final"], s=22, alpha=0.75)
    ax.plot([0, 1], [0, 1], color="black", linewidth=1, linestyle="--")
    ax.set_xlabel("prompt-only commitment")
    ax.set_ylabel("final-prefix commitment")
    ax.set_title("Final-prefix vs prompt-only commitment")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    save_fig(fig, "final_vs_prompt_only_commitment.png")

    display(prompt_only_summary)
else:
    warn("Prompt-only scores are not available; score-dependent prompt-only plots were skipped.")


## 3. Before-Action-Mention Diagnostics

In [ ]:

mention_cols = [
    "is_before_first_action_mention", "is_before_first_final_action_mention",
    "first_action_mention_token_index", "first_final_action_mention_token_index",
]
missing_mention_cols = [c for c in mention_cols if c not in prefix_df.columns]

action_mention_summary = pd.DataFrame()
if missing_mention_cols:
    warn(f"Skipping some before-action-mention diagnostics; missing columns: {missing_mention_cols}")
else:
    before_any = prefix_df[prefix_df["is_before_first_action_mention"].fillna(True)].copy()
    before_final = prefix_df[prefix_df["is_before_first_final_action_mention"].fillna(True)].copy()

    before_any_summary, before_any_counts = binned_per_step_curve(before_any, "commitment_to_final", n_bins=N_BINS)
    before_final_summary, before_final_counts = binned_per_step_curve(before_final, "commitment_to_final", n_bins=N_BINS)
    plot_curve(before_any_summary, "commitment_to_final", "commitment to final action", "Commitment before any explicit action mention", "commitment_progress_before_any_action_mention_sem.png", ylim=(0, 1))
    plot_curve(before_final_summary, "commitment_to_final", "commitment to final action", "Commitment before final-action mention", "commitment_progress_before_final_action_mention_sem.png", ylim=(0, 1))
    plot_counts(before_any_counts, "Coverage before any action mention", "progress_bin_counts_before_any_action_mention.png")
    plot_counts(before_final_counts, "Coverage before final-action mention", "progress_bin_counts_before_final_action_mention.png")

    norm = prefix_df.copy()
    denom = norm["first_action_mention_token_index"].replace({0: np.nan}).astype(float)
    norm["progress_to_first_action_mention"] = norm["prefix_token_index"].astype(float) / denom
    norm = norm[np.isfinite(norm["progress_to_first_action_mention"]) & (norm["progress_to_first_action_mention"] >= 0)].copy()
    norm["progress_to_first_action_mention"] = norm["progress_to_first_action_mention"].clip(0, 1)
    norm_commit_summary, _ = binned_per_step_curve(norm, "commitment_to_final", progress_col="progress_to_first_action_mention", n_bins=N_BINS)
    norm_entropy_summary, _ = binned_per_step_curve(norm, "entropy_commitment", progress_col="progress_to_first_action_mention", n_bins=N_BINS)
    plot_curve(norm_commit_summary, "commitment_to_final", "commitment to final action", "Commitment normalized to first action mention", "commitment_to_first_action_mention_progress.png", ylim=(0, 1))
    plot_curve(norm_entropy_summary, "entropy_commitment", "entropy commitment", "Entropy commitment normalized to first action mention", "entropy_to_first_action_mention_progress.png", ylim=(0, 1))

    per_decision_mentions = prefix_df.sort_values(["decision_id", "prefix_token_index"]).groupby("decision_id", as_index=False).first()
    per_decision_mentions["first_action_mention_progress"] = per_decision_mentions["first_action_mention_token_index"] / per_decision_mentions["num_analysis_tokens"].replace({0: np.nan})
    per_decision_mentions["first_final_action_mention_progress"] = per_decision_mentions["first_final_action_mention_token_index"] / per_decision_mentions["num_analysis_tokens"].replace({0: np.nan})
    action_mention_summary = pd.DataFrame([
        {"metric": "decision_steps", "value": per_decision_mentions["decision_id"].nunique()},
        {"metric": "fraction_steps_with_first_action_mention", "value": per_decision_mentions["first_action_mention_token_index"].notna().mean()},
        {"metric": "median_first_action_mention_progress", "value": per_decision_mentions["first_action_mention_progress"].median()},
        {"metric": "median_first_final_action_mention_progress", "value": per_decision_mentions["first_final_action_mention_progress"].median()},
        {"metric": "steps_with_no_action_mention", "value": per_decision_mentions["first_action_mention_token_index"].isna().sum()},
        {"metric": "steps_final_action_never_explicitly_mentioned", "value": per_decision_mentions["first_final_action_mention_token_index"].isna().sum()},
    ])
    action_mention_summary.to_csv(OUT_DIR / "action_mention_summary.csv", index=False)
    display(action_mention_summary)


## 4. PCA Diagnostics Before Action Mention

In [ ]:

# Enrich manifold rows with action-mention indices/text when the manifold file does not carry them.
manifold_work = manifold_df.copy()
if prefixes_optional is not None and "prefix_id" in manifold_work.columns and "prefix_id" in prefixes_optional.columns:
    add_cols = [c for c in ["first_action_mention_token_index", "first_final_action_mention_token_index", "analysis_prefix_text"] if c in prefixes_optional.columns and c not in manifold_work.columns]
    if add_cols:
        manifold_work = manifold_work.merge(prefixes_optional[["prefix_id"] + add_cols].drop_duplicates("prefix_id"), on="prefix_id", how="left")

if {"beh_pca1", "beh_pca2"}.issubset(manifold_work.columns):
    if "is_before_first_action_mention" in manifold_work.columns:
        m_before_any = manifold_work[manifold_work["is_before_first_action_mention"].fillna(True)].copy()
    else:
        warn("behavior_manifold has no is_before_first_action_mention; using all rows for before-any PCA diagnostic")
        m_before_any = manifold_work.copy()
    if "is_before_first_final_action_mention" in manifold_work.columns:
        m_before_final = manifold_work[manifold_work["is_before_first_final_action_mention"].fillna(True)].copy()
    else:
        warn("behavior_manifold has no is_before_first_final_action_mention; using all rows for before-final PCA diagnostic")
        m_before_final = manifold_work.copy()

    categorical_scatter(m_before_any, "beh_pca1", "beh_pca2", "agent_action", "PCA before any action mention by final action", "pca_before_any_action_mention_by_agent_action.png")
    categorical_scatter(m_before_final, "beh_pca1", "beh_pca2", "agent_action", "PCA before final-action mention by final action", "pca_before_final_action_mention_by_agent_action.png")
    continuous_scatter(m_before_any, "beh_pca1", "beh_pca2", "commitment_to_final", "PCA before any action mention by commitment", "pca_before_any_action_mention_by_commitment.png")
    continuous_scatter(m_before_any, "beh_pca1", "beh_pca2", "entropy_commitment", "PCA before any action mention by entropy commitment", "pca_before_any_action_mention_by_entropy.png")
    continuous_scatter(m_before_any, "beh_pca1", "beh_pca2", "progress", "PCA before any action mention by progress", "pca_before_any_action_mention_by_progress.png")

    pca_before_summary = (
        m_before_any.groupby("agent_action", dropna=False)
        .agg(
            prefix_points=("decision_id", "size"),
            decision_steps=("decision_id", "nunique"),
            mean_commitment_to_final=("commitment_to_final", "mean"),
            mean_entropy_commitment=("entropy_commitment", "mean"),
            mean_progress=("progress", "mean"),
            mean_beh_pca1=("beh_pca1", "mean"),
            mean_beh_pca2=("beh_pca2", "mean"),
        )
        .reset_index()
    )
    pca_before_summary.to_csv(OUT_DIR / "pca_before_action_mention_summary.csv", index=False)
    display(pca_before_summary)
else:
    warn("Skipping PCA diagnostics; behavior_manifold lacks beh_pca1/beh_pca2")
    pca_before_summary = pd.DataFrame()


## 5. Lock-In and Switching

In [ ]:

step_work = step_df.copy()
if "decision_id" not in step_work.columns:
    step_work = ensure_decision_id(step_work)

if not step_work.empty:
    def hist_column(df, col, filename, title, xlabel, bins=20):
        if col not in df.columns or df[col].dropna().empty:
            warn(f"Skipping {filename}; missing/empty {col}")
            return None
        fig, ax = plt.subplots()
        ax.hist(df[col].dropna(), bins=bins, alpha=0.85)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("decision steps")
        ax.set_title(title)
        return save_fig(fig, filename)

    hist_column(step_work, "lock_in_progress", "lock_in_progress_histogram.png", "Lock-in progress", "lock-in progress")

    if {"first_cross_0_5_progress", "first_cross_0_7_progress"}.issubset(step_work.columns):
        fig, ax = plt.subplots()
        ax.hist(step_work["first_cross_0_5_progress"].dropna(), bins=20, alpha=0.65, label="cross 0.5")
        ax.hist(step_work["first_cross_0_7_progress"].dropna(), bins=20, alpha=0.65, label="cross 0.7")
        ax.set_xlabel("progress")
        ax.set_ylabel("decision steps")
        ax.set_title("First commitment threshold crossing")
        ax.legend()
        save_fig(fig, "first_cross_progress_histograms.png")
    else:
        warn("Skipping first_cross_progress_histograms.png; threshold columns missing")

    hist_column(step_work, "switch_count", "switch_count_histogram.png", "Top-action switch count", "switch count", bins=range(0, int(step_work.get("switch_count", pd.Series([0])).max()) + 3))
    hist_column(step_work, "total_variation", "total_variation_histogram.png", "Total variation across prefix probabilities", "total variation")

    if {"switch_count", "final_prefix_commitment_to_final"}.issubset(step_work.columns):
        fig, ax = plt.subplots()
        ax.scatter(step_work["switch_count"], step_work["final_prefix_commitment_to_final"], s=24, alpha=0.75)
        ax.set_xlabel("switch count")
        ax.set_ylabel("final-prefix commitment to final action")
        ax.set_title("Switch count vs final-prefix commitment")
        save_fig(fig, "switch_count_vs_final_commitment.png")

    if {"lock_in_progress", "switch_count"}.issubset(step_work.columns):
        fig, ax = plt.subplots()
        ax.scatter(step_work["lock_in_progress"], step_work["switch_count"], s=24, alpha=0.75)
        ax.set_xlabel("lock-in progress")
        ax.set_ylabel("switch count")
        ax.set_title("Lock-in progress vs switch count")
        save_fig(fig, "lock_in_vs_switch_count.png")

    early_lockin = step_work.dropna(subset=["lock_in_progress"]).sort_values("lock_in_progress").head(20)
    late_lockin = step_work.dropna(subset=["lock_in_progress"]).sort_values("lock_in_progress", ascending=False).head(20)
    no_lockin = step_work[step_work["lock_in_progress"].isna()].head(20) if "lock_in_progress" in step_work.columns else pd.DataFrame()
    high_switch = step_work.sort_values("switch_count", ascending=False).head(20) if "switch_count" in step_work.columns else pd.DataFrame()
    low_final_commitment = step_work.sort_values("final_prefix_commitment_to_final").head(20) if "final_prefix_commitment_to_final" in step_work.columns else pd.DataFrame()

    early_lockin.to_csv(OUT_DIR / "early_lockin_examples.csv", index=False)
    late_lockin.to_csv(OUT_DIR / "late_lockin_examples.csv", index=False)
    no_lockin.to_csv(OUT_DIR / "no_lockin_examples.csv", index=False)
    high_switch.to_csv(OUT_DIR / "high_switch_examples.csv", index=False)
    low_final_commitment.to_csv(OUT_DIR / "low_final_commitment_examples.csv", index=False)

    display(pd.DataFrame({
        "example_set": ["early_lockin", "late_lockin", "no_lockin", "high_switch", "low_final_commitment"],
        "rows": [len(early_lockin), len(late_lockin), len(no_lockin), len(high_switch), len(low_final_commitment)],
    }))
else:
    warn("Skipping lock-in and switching diagnostics because step summary is empty")


## 6. Inspect High-Switch and Late-Lock-In Examples

In [ ]:

def plot_q_curves_for_decision(decision_id, overview_ax=None):
    group = prefix_df[prefix_df["decision_id"] == decision_id].sort_values("prefix_token_index").copy()
    if group.empty:
        return None
    meta = step_work[step_work["decision_id"] == decision_id].head(1)
    if overview_ax is None:
        fig, ax = plt.subplots(figsize=(8.5, 5.0))
    else:
        fig, ax = None, overview_ax
    colors = {"UP": "#4C78A8", "DOWN": "#F58518", "LEFT": "#54A24B", "RIGHT": "#B279A2"}
    for action in ACTIONS:
        ax.plot(group["prefix_token_index"], group[f"q_{action}"], label=action, color=colors[action], linewidth=1.8)
    if "first_action_mention_token_index" in group.columns:
        first_idx = group["first_action_mention_token_index"].dropna()
        if not first_idx.empty:
            ax.axvline(first_idx.iloc[0], color="black", linestyle="--", linewidth=1, alpha=0.75, label="first action mention")
    if "first_final_action_mention_token_index" in group.columns:
        final_idx = group["first_final_action_mention_token_index"].dropna()
        if not final_idx.empty:
            ax.axvline(final_idx.iloc[0], color="gray", linestyle=":", linewidth=1.2, alpha=0.85, label="first final-action mention")
    if not meta.empty and "lock_in_progress" in meta.columns and pd.notna(meta["lock_in_progress"].iloc[0]):
        lock_progress = meta["lock_in_progress"].iloc[0]
        lock_candidates = group[group["progress"] >= lock_progress]
        if not lock_candidates.empty:
            lock_token = lock_candidates["prefix_token_index"].iloc[0]
            ax.axvline(lock_token, color="red", linestyle="-.", linewidth=1.2, alpha=0.8, label="lock-in")
    agent_action = group["agent_action"].iloc[0] if "agent_action" in group.columns else "?"
    switch_count = meta["switch_count"].iloc[0] if not meta.empty and "switch_count" in meta.columns else np.nan
    lock_in = meta["lock_in_progress"].iloc[0] if not meta.empty and "lock_in_progress" in meta.columns else np.nan
    final_commit = meta["final_prefix_commitment_to_final"].iloc[0] if not meta.empty and "final_prefix_commitment_to_final" in meta.columns else np.nan
    ax.set_title(f"{decision_id}\nagent={agent_action}, switches={switch_count}, lock={lock_in}, final_commit={final_commit:.3f}" if pd.notna(final_commit) else str(decision_id), fontsize=9)
    ax.set_xlabel("prefix token index")
    ax.set_ylabel("q(action)")
    ax.set_ylim(0, 1)
    if overview_ax is None:
        ax.legend(loc="best", fontsize=8)
        return save_q_fig(fig, f"{safe_name(decision_id)}.png")
    return ax

selected_ids = []
for candidate_df in [
    step_work.sort_values("switch_count", ascending=False).head(6) if "switch_count" in step_work.columns else pd.DataFrame(),
    step_work.dropna(subset=["lock_in_progress"]).sort_values("lock_in_progress", ascending=False).head(6) if "lock_in_progress" in step_work.columns else pd.DataFrame(),
    step_work.dropna(subset=["lock_in_progress"]).sort_values("lock_in_progress").head(6) if "lock_in_progress" in step_work.columns else pd.DataFrame(),
]:
    if "decision_id" in candidate_df.columns:
        selected_ids.extend(candidate_df["decision_id"].astype(str).tolist())
selected_ids = list(dict.fromkeys(selected_ids))

for decision_id in selected_ids:
    plot_q_curves_for_decision(decision_id)

high_switch_ids = step_work.sort_values("switch_count", ascending=False).head(6)["decision_id"].astype(str).tolist() if "switch_count" in step_work.columns else []
if high_switch_ids:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
    for ax, decision_id in zip(axes.ravel(), high_switch_ids):
        plot_q_curves_for_decision(decision_id, overview_ax=ax)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=6, fontsize=8)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, "q_curves_high_switch_overview.png")
else:
    warn("No high-switch decisions available for overview plot")

text_excerpt_rows = []
if "analysis_prefix_text" in prefix_df.columns and "top_action_changed" in prefix_df.columns:
    for decision_id in high_switch_ids:
        group = prefix_df[prefix_df["decision_id"] == decision_id].sort_values("prefix_token_index").copy()
        switches = group[group["top_action_changed"].fillna(False)]
        if switches.empty:
            continue
        sw = switches.iloc[0]
        before = group[group["prefix_token_index"] < sw["prefix_token_index"]].tail(1)
        text = sw.get("analysis_prefix_text")
        excerpt = "" if pd.isna(text) else str(text)[-800:]
        text_excerpt_rows.append({
            "decision_id": decision_id,
            "agent_action": sw.get("agent_action"),
            "switch_token_index": sw.get("prefix_token_index"),
            "top_action_before": before["top_action"].iloc[0] if not before.empty else None,
            "top_action_after": sw.get("top_action"),
            "text_excerpt": excerpt,
            "first_action_mention_token_index": sw.get("first_action_mention_token_index"),
            "first_final_action_mention_token_index": sw.get("first_final_action_mention_token_index"),
        })
else:
    warn("Skipping high-switch text excerpts; analysis_prefix_text or top_action_changed is missing")

high_switch_text_excerpts = pd.DataFrame(text_excerpt_rows)
high_switch_text_excerpts.to_csv(OUT_DIR / "high_switch_text_excerpts.csv", index=False)
display(high_switch_text_excerpts.head())


## 7. Action Imbalance and CoT Length

In [ ]:

finals = final_prefix_rows(prefix_df)
firsts = first_per_decision(prefix_df)
per_step_mentions = firsts.copy()
if {"first_action_mention_token_index", "num_analysis_tokens"}.issubset(per_step_mentions.columns):
    per_step_mentions["first_action_mention_progress"] = per_step_mentions["first_action_mention_token_index"] / per_step_mentions["num_analysis_tokens"].replace({0: np.nan})
else:
    per_step_mentions["first_action_mention_progress"] = np.nan

step_counts_by_action = finals.groupby("agent_action")["decision_id"].nunique().rename("decision_steps")
prefix_counts_by_action = prefix_df.groupby("agent_action")["decision_id"].size().rename("prefix_points")
length_stats = finals.groupby("agent_action")["num_analysis_tokens"].agg(mean_num_analysis_tokens="mean", median_num_analysis_tokens="median", max_num_analysis_tokens="max") if "num_analysis_tokens" in finals.columns else pd.DataFrame()
mention_stats = per_step_mentions.groupby("agent_action")["first_action_mention_progress"].mean().rename("mean_first_action_mention_progress")
final_commit_stats = finals.groupby("agent_action")["commitment_to_final"].mean().rename("mean_final_commitment_to_final")

action_balance_summary = pd.concat([step_counts_by_action, prefix_counts_by_action, length_stats, mention_stats, final_commit_stats], axis=1).reindex(ACTIONS)
if prompt_only_scores is not None and "prompt_only_commitment_to_final" in prompt_only_scores.columns:
    po_action = prompt_only_scores.groupby("agent_action")["prompt_only_commitment_to_final"].mean().rename("mean_prompt_only_commitment_to_final")
    action_balance_summary = action_balance_summary.join(po_action)
action_balance_summary = action_balance_summary.reset_index().rename(columns={"index": "agent_action"})
action_balance_summary.to_csv(OUT_DIR / "action_balance_summary.csv", index=False)

def bar_action(series, filename, title, ylabel):
    fig, ax = plt.subplots()
    values = series.reindex(ACTIONS).fillna(0)
    ax.bar(values.index, values.values, color="#4C78A8", alpha=0.85)
    ax.set_xlabel("agent_action")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    save_fig(fig, filename)

bar_action(step_counts_by_action, "action_distribution_steps.png", "Action distribution by decision step", "decision steps")
bar_action(prefix_counts_by_action, "action_distribution_prefixes.png", "Action distribution by prefix row", "prefix rows")
if "num_analysis_tokens" in finals.columns:
    fig, ax = plt.subplots()
    data = [finals.loc[finals["agent_action"] == a, "num_analysis_tokens"].dropna().values for a in ACTIONS]
    ax.boxplot(data, labels=ACTIONS, showfliers=False)
    ax.set_xlabel("agent_action")
    ax.set_ylabel("num analysis tokens")
    ax.set_title("CoT length by final action")
    save_fig(fig, "cot_length_by_agent_action.png")
bar_action(final_commit_stats, "final_commitment_by_agent_action.png", "Final-prefix commitment by final action", "mean final commitment")
if per_step_mentions["first_action_mention_progress"].notna().any():
    bar_action(mention_stats, "first_action_mention_progress_by_agent_action.png", "First action mention progress by final action", "mean first mention progress")
else:
    warn("Skipping first_action_mention_progress_by_agent_action.png; no first-action mention progress available")

display(action_balance_summary)


## 8. Cheap Shuffled-Label Control

In [ ]:

rng = np.random.default_rng(20260627)
shuffle_df = prefix_df.copy()
shuffle_df["shuffled_agent_action"] = rng.permutation(shuffle_df["agent_action"].to_numpy())
shuffle_df["commitment_to_shuffled_action"] = q_for_action(shuffle_df, action_col="shuffled_agent_action")
shuffle_df["delta_true_minus_shuffled_commitment"] = shuffle_df["commitment_to_final"] - shuffle_df["commitment_to_shuffled_action"]

true_summary, _ = binned_per_step_curve(shuffle_df, "commitment_to_final", n_bins=N_BINS)
shuf_summary, _ = binned_per_step_curve(shuffle_df, "commitment_to_shuffled_action", n_bins=N_BINS)
delta_shuf_summary, _ = binned_per_step_curve(shuffle_df, "delta_true_minus_shuffled_commitment", n_bins=N_BINS)

fig, ax = plt.subplots()
for summary, label in [(true_summary, "true final action"), (shuf_summary, "shuffled action")]:
    x = summary["progress_mid"].to_numpy(dtype=float)
    y = summary["mean"].to_numpy(dtype=float)
    sem = summary["sem"].fillna(0).to_numpy(dtype=float)
    ax.plot(x, y, marker="o", linewidth=2, label=label)
    ax.fill_between(x, y - sem, y + sem, alpha=0.15)
ax.set_xlabel("progress")
ax.set_ylabel("commitment")
ax.set_ylim(0, 1)
ax.set_title("Shuffled-label control over progress")
ax.legend()
save_fig(fig, "shuffled_label_control_progress.png")

final_shuffle = final_prefix_rows(shuffle_df)
fig, ax = plt.subplots()
ax.hist(final_shuffle["commitment_to_final"].dropna(), bins=20, alpha=0.65, label="true final")
ax.hist(final_shuffle["commitment_to_shuffled_action"].dropna(), bins=20, alpha=0.65, label="shuffled")
ax.set_xlabel("final-prefix commitment")
ax.set_ylabel("decision steps")
ax.set_title("Final-prefix shuffled-label control")
ax.legend()
save_fig(fig, "shuffled_label_final_histogram.png")

plot_curve(delta_shuf_summary, "delta_true_minus_shuffled_commitment", "true minus shuffled commitment", "True-vs-shuffled commitment gap", "shuffled_label_delta_progress.png")

shuffled_label_control_summary = pd.DataFrame([
    {"metric": "mean_prefix_commitment_true", "value": shuffle_df["commitment_to_final"].mean()},
    {"metric": "mean_prefix_commitment_shuffled", "value": shuffle_df["commitment_to_shuffled_action"].mean()},
    {"metric": "mean_prefix_true_minus_shuffled", "value": shuffle_df["delta_true_minus_shuffled_commitment"].mean()},
    {"metric": "mean_final_commitment_true", "value": final_shuffle["commitment_to_final"].mean()},
    {"metric": "mean_final_commitment_shuffled", "value": final_shuffle["commitment_to_shuffled_action"].mean()},
    {"metric": "mean_final_true_minus_shuffled", "value": (final_shuffle["commitment_to_final"] - final_shuffle["commitment_to_shuffled_action"]).mean()},
])
shuffled_label_control_summary.to_csv(OUT_DIR / "shuffled_label_control_summary.csv", index=False)
display(shuffled_label_control_summary)


## 9. Optional Mismatched Prompt/CoT Control Preparation

In [ ]:

MISMATCH_PREFIX_PATH = OUT_DIR / "mismatched_prompt_cot_prefixes.parquet"
MISMATCH_SCORE_PATH = OUT_DIR / "mismatched_prompt_cot_scores.parquet"
mismatch_command = """python scripts/03_score_prefix_actions_together.py \\
  --input outputs/diagnostics/mismatched_prompt_cot_prefixes.parquet \\
  --output outputs/diagnostics/mismatched_prompt_cot_scores.parquet \\
  --model_name openai/gpt-oss-20b \\
  --api_key_env TOGETHER_API_KEY \\
  --scoring_method echo_teacher_forced \\
  --candidate_mode bare_action \\
  --temperature 0.0 \\
  --top_p 1.0 \\
  --logprobs 20 \\
  --max_tokens 1 \\
  --save_every 50 \\
  --resume"""

mismatch_summary = pd.DataFrame()
if prefixes_optional is not None and {"decision_id", "trajectory_id", "step_id", "prompt_text", "analysis_prefix_text", "agent_action", "progress"}.issubset(prefixes_optional.columns):
    source = prefixes_optional[prefixes_optional["decision_id"].isin(prefix_df["decision_id"].unique())].copy()
    if source.empty:
        warn("No overlapping decisions between current scores and outputs/prefixes.parquet; cannot create mismatch prefixes.")
    else:
        selected_rows = []
        for decision_id, group in source.groupby("decision_id"):
            group = group.sort_values("progress")
            for target_progress, label in [(0.1, "early"), (0.5, "mid"), (0.9, "late")]:
                idx = (group["progress"] - target_progress).abs().idxmin()
                row = group.loc[idx].copy()
                row["mismatch_position"] = label
                selected_rows.append(row)
        selected = pd.DataFrame(selected_rows).reset_index(drop=True)
        rng = np.random.default_rng(20260627)
        output_rows = []
        for _, row in selected.iterrows():
            donor_pool = selected[selected["agent_action"] != row["agent_action"]]
            if donor_pool.empty:
                continue
            donor = donor_pool.iloc[int(rng.integers(0, len(donor_pool)))]
            out = row.copy()
            out["prefix_id"] = f"{row['decision_id']}::mismatch::{row['mismatch_position']}"
            out["analysis_prefix_text"] = donor["analysis_prefix_text"]
            if "analysis_prefix_token_ids_json" in donor.index:
                out["analysis_prefix_token_ids_json"] = donor.get("analysis_prefix_token_ids_json")
            out["mismatch_source_decision_id"] = donor["decision_id"]
            out["mismatch_source_agent_action"] = donor["agent_action"]
            out["mismatch_source_prefix_token_index"] = donor.get("prefix_token_index")
            out["mismatch_source_progress"] = donor.get("progress")
            output_rows.append(out)
        mismatch_prefixes = pd.DataFrame(output_rows)
        mismatch_prefixes.to_parquet(MISMATCH_PREFIX_PATH, index=False)
        print(f"Created {MISMATCH_PREFIX_PATH} with {len(mismatch_prefixes)} rows.")
        print("Score mismatched prompt/CoT prefixes explicitly with:")
        print(mismatch_command)
else:
    warn("Cannot create mismatched prompt/CoT prefixes because outputs/prefixes.parquet lacks required columns.")
    print("Expected scoring command if the control parquet is created later:")
    print(mismatch_command)

if MISMATCH_SCORE_PATH.exists():
    mismatch_scores = ensure_decision_id(pd.read_parquet(MISMATCH_SCORE_PATH))
    if set(Q_COLS + ["agent_action", "mismatch_source_agent_action"]).issubset(mismatch_scores.columns):
        mismatch_scores["commitment_to_prompt_action"] = q_for_action(mismatch_scores, action_col="agent_action")
        mismatch_scores["commitment_to_cot_source_action"] = q_for_action(mismatch_scores, action_col="mismatch_source_agent_action")
        mismatch_scores["mismatch_preference"] = np.select(
            [
                mismatch_scores["commitment_to_prompt_action"] > mismatch_scores["commitment_to_cot_source_action"],
                mismatch_scores["commitment_to_prompt_action"] < mismatch_scores["commitment_to_cot_source_action"],
            ],
            ["prompt", "cot_source"],
            default="tie",
        )
        mismatch_summary = (
            mismatch_scores.groupby("mismatch_position", dropna=False)
            .agg(
                rows=("decision_id", "size"),
                mean_commitment_to_prompt_action=("commitment_to_prompt_action", "mean"),
                mean_commitment_to_cot_source_action=("commitment_to_cot_source_action", "mean"),
                fraction_prompt_preferred=("mismatch_preference", lambda x: (x == "prompt").mean()),
                fraction_cot_source_preferred=("mismatch_preference", lambda x: (x == "cot_source").mean()),
            )
            .reset_index()
        )
        mismatch_summary.to_csv(OUT_DIR / "mismatched_prompt_cot_summary.csv", index=False)
        fig, ax = plt.subplots()
        x = np.arange(len(mismatch_summary))
        width = 0.36
        ax.bar(x - width/2, mismatch_summary["mean_commitment_to_prompt_action"], width, label="prompt action")
        ax.bar(x + width/2, mismatch_summary["mean_commitment_to_cot_source_action"], width, label="CoT source action")
        ax.set_xticks(x)
        ax.set_xticklabels(mismatch_summary["mismatch_position"].astype(str))
        ax.set_ylim(0, 1)
        ax.set_ylabel("mean commitment")
        ax.set_title("Mismatched prompt/CoT control")
        ax.legend()
        save_fig(fig, "mismatched_prompt_cot_control.png")
        display(mismatch_summary)
    else:
        warn("Mismatched score file exists but lacks required q/action columns")
else:
    print("No mismatched score file found; API-dependent mismatch summary skipped.")


## 10. Final Report

In [ ]:

def fmt(value, digits=3):
    if value is None or (isinstance(value, float) and not np.isfinite(value)) or pd.isna(value):
        return "NA"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.{digits}f}"
    return str(value)

report = []
report.append("# Behavior Diagnostics Report")
report.append("")
report.append("## Dataset")
report.append(f"- Prefix points: {len(prefix_df):,}")
report.append(f"- Decision steps: {prefix_df['decision_id'].nunique():,}")
report.append(f"- Trajectory files: {prefix_df['trajectory_id'].nunique() if 'trajectory_id' in prefix_df.columns else 'NA'}")
report.append(f"- Prefixes per decision, median/mean/max: {fmt(prefix_counts.median())} / {fmt(prefix_counts.mean())} / {fmt(prefix_counts.max())}")
report.append("")
report.append("Agent-action counts:")
for action, count in action_distribution(final_prefix_rows(prefix_df)["agent_action"]).items():
    report.append(f"- {action}: {int(count)} decision steps")

report.append("")
report.append("## Main Replication Curves")
if not commit_summary.empty:
    first_commit = commit_summary["mean"].dropna().iloc[0] if not commit_summary["mean"].dropna().empty else np.nan
    last_commit = commit_summary["mean"].dropna().iloc[-1] if not commit_summary["mean"].dropna().empty else np.nan
    report.append(f"- Commitment to final action changes from {fmt(first_commit)} to {fmt(last_commit)} across progress bins.")
if not entropy_summary.empty:
    first_entropy = entropy_summary["mean"].dropna().iloc[0] if not entropy_summary["mean"].dropna().empty else np.nan
    last_entropy = entropy_summary["mean"].dropna().iloc[-1] if not entropy_summary["mean"].dropna().empty else np.nan
    report.append(f"- Entropy commitment changes from {fmt(first_entropy)} to {fmt(last_entropy)} across progress bins.")
report.append("- Per-step weighted curve table: `outputs/diagnostics/progress_curve_per_step_weighted.csv`.")

report.append("")
report.append("## Prompt-Only Baseline")
if prompt_only_scores is not None and not prompt_only_summary.empty:
    report.append(f"- Source: {prompt_only_source}.")
    for _, row in prompt_only_summary.iterrows():
        if row["metric"] != "source":
            report.append(f"- {row['metric']}: {fmt(row['value'])}")
else:
    report.append("- Prompt-only scores were not available. The notebook created or described the opt-in scoring command without calling the API.")

report.append("")
report.append("## Before Action Mention")
if not action_mention_summary.empty:
    for _, row in action_mention_summary.iterrows():
        report.append(f"- {row['metric']}: {fmt(row['value'])}")
else:
    report.append("- Before-action-mention diagnostics were skipped because required columns were unavailable.")

report.append("")
report.append("## Manifold Diagnostics")
if 'pca_before_summary' in globals() and not pca_before_summary.empty:
    report.append("- PCA before-action summaries were written to `outputs/diagnostics/pca_before_action_mention_summary.csv`.")
    if {"beh_pca1", "beh_pca2", "commitment_to_final"}.issubset(manifold_work.columns):
        radius_work = manifold_work.copy()
        radius_work["pca_radius"] = np.sqrt(radius_work["beh_pca1"] ** 2 + radius_work["beh_pca2"] ** 2)
        low = radius_work[radius_work["commitment_to_final"] <= radius_work["commitment_to_final"].quantile(0.25)]["pca_radius"].mean()
        high = radius_work[radius_work["commitment_to_final"] >= radius_work["commitment_to_final"].quantile(0.75)]["pca_radius"].mean()
        report.append(f"- Mean PCA radius for low vs high commitment quartiles: {fmt(low)} vs {fmt(high)}. Treat this as descriptive, not causal.")
else:
    report.append("- PCA diagnostics were skipped or empty.")

report.append("")
report.append("## Switching and Lock-In")
if not step_work.empty:
    report.append(f"- Mean switch count: {fmt(step_work['switch_count'].mean() if 'switch_count' in step_work.columns else np.nan)}")
    report.append(f"- Median lock-in progress: {fmt(step_work['lock_in_progress'].median() if 'lock_in_progress' in step_work.columns else np.nan)}")
    report.append(f"- Mean final-prefix commitment: {fmt(step_work['final_prefix_commitment_to_final'].mean() if 'final_prefix_commitment_to_final' in step_work.columns else np.nan)}")
    if "final_prefix_is_top_action_final" in step_work.columns:
        report.append(f"- Fraction of final prefixes where top_action equals agent_action: {fmt(step_work['final_prefix_is_top_action_final'].mean())}")
    report.append("- Example tables were written for early lock-in, late lock-in, no lock-in, high-switch, and low-final-commitment decisions.")
else:
    report.append("- Step summary was empty; switching and lock-in diagnostics were skipped.")

report.append("")
report.append("## Controls")
if 'shuffled_label_control_summary' in globals() and not shuffled_label_control_summary.empty:
    for _, row in shuffled_label_control_summary.iterrows():
        report.append(f"- {row['metric']}: {fmt(row['value'])}")
if mismatch_summary is not None and not mismatch_summary.empty:
    report.append("- Mismatched prompt/CoT score summary is available in `outputs/diagnostics/mismatched_prompt_cot_summary.csv`.")
else:
    report.append("- Mismatched prompt/CoT prefixes were prepared where possible; scoring remains opt-in and was not called by this notebook.")

report.append("")
report.append("## Cautious Interpretation")
report.append("- Rising commitment over progress is consistent with action probability mass consolidating during generated reasoning, but it does not by itself prove that the text caused the decision.")
report.append("- Before-action-mention curves are the key robustness check for whether action commitment is visible before explicit action tokens appear in the analysis text.")
report.append("- Prompt-only and mismatched prompt/CoT controls should be interpreted with the exact scoring template in mind; they test the scoring probe, not the full policy execution path.")
report.append("- The current diagnostics are descriptive and should be paired with held-out trajectories or additional seeds before making strong claims.")

if WARNINGS:
    report.append("")
    report.append("## Warnings")
    for msg in WARNINGS:
        report.append(f"- {msg}")

report_text = "\n".join(report) + "\n"
report_path = OUT_DIR / "behavior_diagnostics_report.md"
report_path.write_text(report_text)
print(report_text)
print(f"Wrote {report_path}")
